# Render the Molecule, in Google Colab

**by [Vishal Bhoir](https://linktr.ee/thebioway)** · [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.23960655.svg)](https://doi.org/10.5281/zenodo.23960655)

Publication-quality images of any Protein Data Bank entry, rendered with Blender and
Molecular Nodes on Google's free computers. Prefer a website? Use the
**[web app](https://vishalbhoir18--render-the-molecule.us-east.modal.run)**.

**How to use**

1. Fill in the form below: your name and email, the PDB ID and the options.
2. Click **Runtime → Run all** (or press Ctrl+F9).
3. The first run installs Blender and Molecular Nodes (about 1–2 minutes). Each render then
   takes about 30–120 seconds.
4. The image appears under step 3 with how to cite it. Run step 4 to download it.

To render another molecule, change the form and run steps 1 and 3 again; the installation is
kept until the Colab session ends.

*Privacy: your name, email and what you render are recorded privately by the site owner to
see who uses this tool. They are not shared.*

In [ ]:
#@title 1. Your details and the molecule { display-mode: "form" }
name = ""  #@param {type:"string"}
email = ""  #@param {type:"string"}
pdb_id = "4HHB"  #@param {type:"string"}
#@markdown ---
style = "cartoon"  #@param ["cartoon", "ribbon", "surface", "spheres", "ball_and_stick", "sticks"]
color = "default"  #@param ["default", "rainbow", "secondary_structure", "goodsell"]
material = "ambient_occlusion"  #@param ["ambient_occlusion", "default", "flat", "squishy"]
viewpoint = "default"  #@param ["default", "front", "back", "top", "bottom", "left", "right"]
resolution = "1280x720"  #@param ["1280x720", "1920x1080", "1080x1080", "800x800"]
quality = "standard"  #@param ["draft", "standard", "high"]
background = "white"  #@param ["white", "transparent", "black"]
ligands = True  #@param {type:"boolean"}
assembly = False  #@param {type:"boolean"}
credit = True  #@param {type:"boolean"}
#@markdown *Colour* "default" = by chain. *Ligands*: show ligands and ions as ball-and-stick
#@markdown (cartoon and ribbon). *Assembly*: the whole biological complex. *Credit*: small
#@markdown credit line on the image (untick for a clean figure and cite in the text).

form = dict(name=name, email=email, pdb_id=pdb_id, style=style, color=color,
            material=material, viewpoint=viewpoint, resolution=resolution, quality=quality,
            background=background, ligands=ligands, assembly=assembly, credit=credit)
import re
if not name.strip() or not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email.strip()) or not pdb_id.strip():
    raise SystemExit("Please fill in your name, a valid email and a PDB ID in the form above, then run again.")
print("Form OK:", pdb_id.strip().upper())

In [ ]:
#@title 2. Install Blender and Molecular Nodes (first run only, about 1–2 minutes) { display-mode: "form" }
import os, subprocess, sys
REPO_URL = "https://github.com/vishalbhoir18/renderthemolecule"
REPO_DIR = "/content/renderthemolecule"
VENV = "/content/rtm-venv"
PYTHON = f"{VENV}/bin/python"
INSTALLED = f"{VENV}/.installed"

def sh(command):
    subprocess.run(command, shell=True, check=True)

if not os.path.exists(INSTALLED):
    print("Installing Blender and Molecular Nodes (about 1–2 minutes)…")
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends "
       "libegl1 libegl-mesa0 libgl1 libgl1-mesa-dri libgomp1 libsm6 libx11-6 libxext6 libxfixes3 libxi6 libxkbcommon0 libxrender1 libxxf86vm1 > /dev/null")
    sh("pip -q install uv")
    sh(f"uv venv -q --python 3.13 {VENV}")
# Always use the latest version of the code (fixes reach open sessions too).
if os.path.exists(REPO_DIR):
    sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin main && git -C {REPO_DIR} reset -q --hard FETCH_HEAD")
else:
    sh(f"git clone -q --depth 1 {REPO_URL} {REPO_DIR}")
sh(f"uv pip install -q --python {PYTHON} -r {REPO_DIR}/requirements.txt")  # quick when up to date
open(INSTALLED, "w").close()
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import importlib
from app import colab
colab = importlib.reload(colab)
print("Ready.")

In [ ]:
#@title 3. Render { display-mode: "form" }
from IPython.display import Image, Markdown, display
OUT_DIR = "/content/render-output"
result = colab.render(form, python=PYTHON, repo_dir=REPO_DIR, out_dir=OUT_DIR)
print(colab.record(result, f"{REPO_DIR}/notebooks/colab_recording.json"))
if result["ok"]:
    display(Image(filename=result["image"]))
    display(Markdown(colab.citation_markdown(result)))
else:
    print("Error:", result["error"])

In [ ]:
#@title 4. Download the image { display-mode: "form" }
from google.colab import files
if result.get("ok"):
    files.download(result["image"])
else:
    print("There is no image to download; see the error in step 3.")

---
Render the Molecule App developed by [Vishal Bhoir](https://linktr.ee/thebioway)
([https://doi.org/10.5281/zenodo.23960655](https://doi.org/10.5281/zenodo.23960655)). Images are rendered with
[Molecular Nodes](https://github.com/BradyAJohnston/MolecularNodes) by Brady Johnston and
contributors, and [Blender](https://www.blender.org). Structures come from the
[RCSB Protein Data Bank](https://www.rcsb.org). Free software under the GNU GPL v3:
[source code](https://github.com/vishalbhoir18/renderthemolecule).